Step 2: Data Preparation

In [ ]:
import re
import pandas as pd
from pathlib import Path

pd.set_option('display.width', 120)

DATA, OUT = Path('data'), Path('out')
OUT.mkdir(exist_ok=True)

results = pd.read_csv(DATA / 'results.csv', na_values=['\\N'])
races   = pd.read_csv(DATA / 'races.csv',   na_values=['\\N'])
status  = pd.read_csv(DATA / 'status.csv',  na_values=['\\N'])

print(f'{len(results):,} result rows loaded')

Log how many rows were removed during each filtering step for auditability.

In [ ]:
audit = []

def step(label, df, note=''):
    prev = audit[-1]['rows'] if audit else len(df)
    audit.append({'step': label, 'rows': len(df),
                  'removed': prev - len(df), 'note': note})
    print(f'{label:<38} {len(df):>7,} rows  ({prev - len(df):+,})  {note}')
    return df

step('loaded results', results);

Join races (for year) and status (for the readable label).

In [ ]:
df = results.merge(races[['raceId', 'year', 'round', 'name']], on='raceId', how='left')
assert df['year'].notna().all(), 'some raceIds had no matching race'
step('joined races (year)', df)

df = df.merge(status, on='statusId', how='left')
assert df['status'].notna().all(), 'some statusIds had no matching status'
step('joined status', df);

Filter to 2009 onward and assign era.

In [ ]:
df = df[df['year'] >= 2009].copy()
step('filtered to 2009+', df, 'pre-2009 out of scope')

def to_era(y):
    if 2009 <= y <= 2010: return '1. Pre-DRS (2009-10)'
    if 2011 <= y <= 2013: return '2. DRS, V8 (2011-13)'
    if 2014 <= y <= 2021: return '3. Hybrid (2014-21)'
    if y >= 2022:         return '4. Ground effect (2022+)'
    return None

df['era'] = df['year'].apply(to_era)
assert df['era'].notna().all(), 'a year fell outside every era'
print()
print(df.groupby('era')['year'].agg(['min', 'max', 'size']).to_string())

Drop pit-lane starts. grid == 0 means the car started from the pit lane, rather than a grid position.

In [ ]:
pit = int((df['grid'] == 0).sum())
df = df[df['grid'] != 0].copy()
step('dropped grid == 0', df, f'{pit} pit-lane starts');

Classify finishers from the actual status strings.

In [ ]:
LAPPED = re.compile(r'^\+\d+\s+Lap', re.IGNORECASE)

finish_labels = sorted({s for s in df['status'].unique()
                        if s == 'Finished' or LAPPED.match(str(s))})
retire_labels = sorted(set(df['status'].unique()) - set(finish_labels))

print('CLASSIFIED FINISH :', finish_labels)
print()
print('RETIREMENT        :', retire_labels)

df['classified'] = df['status'].isin(finish_labels)
print()
print(df['classified'].value_counts().rename({True: 'finished', False: 'retired'}).to_string())

Build dependent variable.

In [ ]:
df['positions_gained'] = df['grid'] - df['positionOrder']
df[['grid', 'positionOrder', 'positions_gained', 'status', 'classified']].head(8)

Zero-sum check. Within a race, grid and positionOrder are permutations of 1..N, so the positions gained must sum to exactly zero: every place someone gains, someone else loses. A mean near zero here confirms the join is correct and the variable is built right. (This only works before filtering. After filtering retirements out, the mean becomes positive.)

In [ ]:
m_all = df['positions_gained'].mean()
print(f'mean positions_gained, everyone : {m_all:+.4f}   <- should be ~0')
assert abs(m_all) < 0.5, 'zero-sum violated: check the join or the grid filter'

Split off finishers.

In [ ]:
full = df.copy()                        # retirements included
df = df[df['classified']].copy()        # primary analysis set
step('kept classified finishers', df, f'{len(full) - len(df):,} retirements removed')

m_fin = df['positions_gained'].mean()
print()
print(f'mean positions_gained, finishers only : {m_fin:+.4f}')

Rows per era.

In [ ]:
counts = (df.groupby('era')
            .agg(driver_races=('positions_gained', 'size'),
                 mean_gained=('positions_gained', 'mean'),
                 sd=('positions_gained', 'std'),
                 seasons=('year', 'nunique'))
            .round(3))
print(counts.to_string())

thin = counts[counts['driver_races'] < 100]
print()
print('all eras have adequate n' if thin.empty else f'THIN: {list(thin.index)}')

Save files.

In [ ]:
keep = ['resultId', 'raceId', 'year', 'round', 'name', 'era', 'driverId',
        'constructorId', 'grid', 'positionOrder', 'positions_gained',
        'status', 'classified']

df[keep].to_csv(OUT / 'analysis_dataset.csv', index=False)
full[keep].to_csv(OUT / 'analysis_dataset_with_retirements.csv', index=False)
pd.DataFrame(audit).to_csv(OUT / 'preparation_audit.csv', index=False)

print(f"analysis_dataset.csv                 {len(df):>7,} rows")
print(f"analysis_dataset_with_retirements.csv {len(full):>7,} rows")
print()
print(pd.DataFrame(audit).to_string(index=False))